<a href="https://colab.research.google.com/github/AnjanaGunarathna/DL_News_Detection/blob/Akeel(DistilBERT)/Fakenews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#pip install transformers datasets evaluate pandas scikit-learn tensorflow

# New Section

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_data():
    # Load the datasets
    fake_news = pd.read_csv("/content/drive/MyDrive/Fake.csv/Fake.csv")
    true_news = pd.read_csv("/content/drive/MyDrive/True.csv/True.csv")

    # Add labels
    true_news["label"] = 1  # 1 for real news
    fake_news["label"] = 0  # 0 for fake news

    # Combine the datasets
    df = pd.concat([true_news, fake_news], ignore_index=True)

    # Shuffle the dataset
    df = df.sample(frac=1).reset_index(drop=True)

    return df

def split_data(df):
    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

# Load and split the data
df = load_data()
X_train, X_test, y_train, y_test = split_data(df)

In [4]:
df.head()

,title,text,subject,date,label
0,Lebanon will only survive if Hezbollah disarms...,ROME (Reuters) - Saudi Arabia s foreign minist...,worldnews,"December 1, 2017",1
1,Myanmar army chief says Rohingya Muslims 'not ...,YANGON (Reuters) - Rohingya Muslims are not na...,worldnews,"October 12, 2017",1
2,Sean Hannity Just Openly Threatened Someone O...,Is there some reason people like Donald Trump ...,News,"April 30, 2017",0
3,COLLEGE QB Kneed Out Of Anger Over Trump’s Rem...,He was given the option to kneel in protest be...,left-news,"Oct 14, 2017",0
4,Honduras opposition parties ask for disputed e...,TEGUCIGALPA (Reuters) - Honduras two main opp...,worldnews,"December 9, 2017",1


In [ ]:
from transformers import DistilBertTokenizer

def tokenize_data(X_train, X_test):
    # Load the DistilBERT tokenizer
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

    # Tokenize the training and testing data
    train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=128)
    test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=128)

    return train_encodings, test_encodings, tokenizer

# Tokenize the data
train_encodings, test_encodings, tokenizer = tokenize_data(X_train, X_test)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import tensorflow as tf

def create_tf_datasets(train_encodings, test_encodings, y_train, y_test):
    # Convert to TensorFlow datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((
        {key: tf.constant(val) for key, val in train_encodings.items()},
        tf.constant(y_train)
    ))
    test_dataset = tf.data.Dataset.from_tensor_slices((
        {key: tf.constant(val) for key, val in test_encodings.items()},
        tf.constant(y_test)
    ))

    # Batch and shuffle the datasets
    train_dataset = train_dataset.shuffle(1000).batch(16)
    test_dataset = test_dataset.batch(16)

    return train_dataset, test_dataset

# Create TensorFlow datasets
train_dataset, test_dataset = create_tf_datasets(train_encodings, test_encodings, y_train, y_test)

In [ ]:
from transformers import TFDistilBertForSequenceClassification
from sklearn.metrics import classification_report

def train_and_evaluate(train_dataset, test_dataset, y_test):
    # Load the pre-trained DistilBERT model
    model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2, from_pt=True)

    # Compile the model
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

    model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

    # Train the model
    history = model.fit(train_dataset, epochs=3, validation_data=test_dataset)

    # Evaluate the model
    results = model.evaluate(test_dataset)
    print("Test Accuracy:", results[1])

    # Generate predictions
    predictions = model.predict(test_dataset)
    predicted_labels = tf.argmax(predictions.logits, axis=1)

    # Print classification report
    print(classification_report(y_test, predicted_labels.numpy()))

    return model

# Train and evaluate the model
model = train_and_evaluate(train_dataset, test_dataset, y_test)

In [ ]:
import os

def save_model(model, tokenizer, save_dir="fine_tuned_distilbert_model"):
    # Save the model and tokenizer
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Model and tokenizer saved to {save_dir}")

# Save the model
save_model(model, tokenizer)


In [ ]:
import shutil

# Replace with your directory name if different
shutil.make_archive('fine_tuned_distilbert_model', 'zip', 'fine_tuned_distilbert_model')

In [ ]:
# Clean ALL .ipynb files under MyDrive so GitHub preview works
import glob, subprocess, shlex

targets = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)
print("Found notebooks:", len(targets))

ok, fail = 0, 0
for p in targets:
    try:
        cmd = f'jupyter nbconvert "{p}" --to notebook --inplace --ClearWidgetsStatePreprocessor.enabled=True'
        subprocess.run(shlex.split(cmd), check=True, capture_output=True)
        ok += 1
    except Exception as e:
        print("Failed:", p, "-", e)
        fail += 1

print(f"Cleaned ✅ {ok}, Failed ❌ {fail}")
